In [ ]:
# -------------------------------------------------------
# CELL 1 - Data Quality Framework
# -------------------------------------------------------
from datetime import datetime, timezone
from pyspark.sql.functions import col, count, when, isnan

PIPELINE_RUN_ID = "data_quality_check"
CHECK_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

print(f"Data Quality Run: {CHECK_TIMESTAMP}")

# -------------------------------------------------------
# Quality check results tracker
# -------------------------------------------------------
results = []

def log_check(table, check, passed, details):
    status = "PASS" if passed else "FAIL"
    results.append({
        "table": table,
        "check": check,
        "status": status,
        "details": details,
        "checked_at": CHECK_TIMESTAMP
    })
    print(f"[{status}] {table} — {check}: {details}")

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

In [ ]:
# -------------------------------------------------------
# CELL 2 - Bronze Layer Quality Checks
# -------------------------------------------------------
# Check 1 - Row count minimums
for table, min_rows in [
    ("bronze_otx_pulses", 100),
    ("bronze_otx_indicators", 1000),
    ("bronze_firewall_logs", 500)
]:
    count = spark.sql(f"SELECT COUNT(*) as total FROM {table}").collect()[0]["total"]
    log_check(table, "minimum_row_count", count >= min_rows, 
              f"{count} rows (minimum {min_rows})")

# Check 2 - No duplicate pipeline runs landing same data
for table in ["bronze_otx_pulses", "bronze_otx_indicators", "bronze_firewall_logs"]:
    dup_count = spark.sql(f"""
        SELECT COUNT(*) as total FROM (
            SELECT pipeline_run_id, COUNT(*) as cnt 
            FROM {table} 
            GROUP BY pipeline_run_id
            HAVING COUNT(*) > 1
        )
    """).collect()[0]["total"]
    log_check(table, "duplicate_pipeline_runs", dup_count >= 0, 
              f"{dup_count} pipeline run(s) found")

In [ ]:
# -------------------------------------------------------
# CELL 3 - Silver Layer Quality Checks
# -------------------------------------------------------
# Check 1 - Null checks on critical fields
silver_null_checks = [
    ("silver_otx_pulses", "pulse_id"),
    ("silver_otx_pulses", "pulse_name"),
    ("silver_otx_indicators", "indicator"),
    ("silver_otx_indicators", "pulse_id"),
    ("silver_firewall_logs", "log_id"),
    ("silver_firewall_logs", "destination"),
    ("silver_firewall_logs", "timestamp")
]

for table, column in silver_null_checks:
    null_count = spark.sql(f"""
        SELECT COUNT(*) as total FROM {table}
        WHERE {column} IS NULL OR {column} = ''
    """).collect()[0]["total"]
    log_check(table, f"null_check_{column}", null_count == 0,
              f"{null_count} nulls in {column}")

# Check 2 - Silver row counts match Bronze
for bronze, silver in [
    ("bronze_otx_pulses", "silver_otx_pulses"),
    ("bronze_otx_indicators", "silver_otx_indicators"),
    ("bronze_firewall_logs", "silver_firewall_logs")
]:
    bronze_count = spark.sql(f"SELECT COUNT(*) as total FROM {bronze}").collect()[0]["total"]
    silver_count = spark.sql(f"SELECT COUNT(*) as total FROM {silver}").collect()[0]["total"]
    log_check(silver, "row_count_matches_bronze", bronze_count == silver_count,
              f"Bronze: {bronze_count} Silver: {silver_count}")

In [ ]:
# -------------------------------------------------------
# CELL 4 - Gold Layer Quality Checks
# -------------------------------------------------------
# Check 1 - Fact table has correlated events
fact_count = spark.sql("SELECT COUNT(*) as total FROM fact_threat_events").collect()[0]["total"]
log_check("fact_threat_events", "has_correlated_events", fact_count > 0,
          f"{fact_count} correlated events")

# Check 2 - All dimension tables populated
for dim, min_rows in [
    ("dim_date", 365),
    ("dim_threat", 100),
    ("dim_indicator", 1000),
    ("dim_department", 5)
]:
    count = spark.sql(f"SELECT COUNT(*) as total FROM {dim}").collect()[0]["total"]
    log_check(dim, "minimum_row_count", count >= min_rows,
              f"{count} rows (minimum {min_rows})")

# Check 3 - No orphaned fact records
orphan_count = spark.sql("""
    SELECT COUNT(*) as total FROM fact_threat_events f
    LEFT JOIN dim_threat t ON f.threat_key = t.threat_key
    WHERE t.threat_key IS NULL
""").collect()[0]["total"]
log_check("fact_threat_events", "no_orphaned_threat_keys", orphan_count == 0,
          f"{orphan_count} orphaned records")

In [ ]:
# -------------------------------------------------------
# CELL 5 - Quality Check Summary
# -------------------------------------------------------
df_results = spark.createDataFrame(results)

passed = df_results.filter(col("status") == "PASS").count()
failed = df_results.filter(col("status") == "FAIL").count()
total = df_results.count()

print(f"\n{'='*50}")
print(f"DATA QUALITY SUMMARY")
print(f"{'='*50}")
print(f"Total checks: {total}")
print(f"Passed: {passed}")
print(f"Failed: {failed}")
print(f"{'='*50}\n")

if failed > 0:
    print("FAILED CHECKS:")
    df_results.filter(col("status") == "FAIL").show(truncate=False)

# Write results to Delta table for tracking
(df_results.write
    .format("delta")
    .mode("append")
    .saveAsTable("data_quality_log")
)

print(f"Results written to data_quality_log")